# Part 5 · Integrated Waste Management Assistant\n**Owner: Teddy**  ·  Part 5 of the EcoSort Waste Management System\n\nOne entry point: image OR text -> classify -> grounded recycling instructions.

### Environment setup — run first (works in VS Code **and** Google Colab)
Detects where you're running, gets the dataset onto **local disk** (fast — no Google Drive), and unzips it.
In Colab it also clones the repo and installs requirements. Set `DATASET_URL` to the `realwaste.zip` GitHub **Release** link once the repo exists.

In [ ]:
import os, sys, zipfile, pathlib, shutil, subprocess

IN_COLAB = "google.colab" in sys.modules
DATASET_URL = ""  # <-- paste the GitHub Release asset URL for realwaste.zip

if IN_COLAB:
    if not os.path.exists("Ecosort-Waste-Management-System"):
        subprocess.run(["git","clone","https://github.com/Ted-star7/Ecosort-Waste-Management-System.git"], check=False)
    if os.path.isdir("Ecosort-Waste-Management-System"):
        os.chdir("Ecosort-Waste-Management-System")
    subprocess.run(["pip","-q","install","-r","requirements.txt"], check=False)
    if DATASET_URL and not os.path.exists("data/realwaste.zip"):
        subprocess.run(["wget","-q","-O","data/realwaste.zip",DATASET_URL], check=False)

# Unzip RealWaste -> data/RealWaste/<category>/  (idempotent)
if not pathlib.Path("data/RealWaste").exists() and pathlib.Path("data/realwaste.zip").exists():
    with zipfile.ZipFile("data/realwaste.zip") as z:
        z.extractall("data/_tmp")
    for p in pathlib.Path("data/_tmp").rglob("RealWaste"):
        shutil.move(str(p), "data/RealWaste"); break
    shutil.rmtree("data/_tmp", ignore_errors=True)

DATA_DIR = "data"
IMAGE_DIR = os.path.join(DATA_DIR, "RealWaste")
print("Setup OK. Images present:", os.path.isdir(IMAGE_DIR))


In [ ]:
import os, re, json, random, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
np.random.seed(42); random.seed(42)

# The 9 categories are read from the folder names (sorted) so every part
# uses ONE identical, consistent label order:
IMG_HEIGHT = IMG_WIDTH = 224
BATCH_SIZE = 32

Requires the functions from Parts 2-4 to be defined (run the master notebook top-to-bottom, or import saved models).

In [ ]:
def waste_management_assistant(input_data, input_type="image"):
    """
    input_type="image" -> classify_waste_image (Part 2)
    input_type="text"  -> classify_waste_description (Part 3)
    then generate_recycling_instructions (Part 4).
    Returns dict with category + instructions + policy sources.
    """
    try:
        if input_type == "image":
            category = classify_waste_image(input_data)
        elif input_type == "text":
            category = classify_waste_description(input_data)
        else:
            return {"ok": False, "error": f"Unknown input_type '{input_type}'"}
        if category not in class_names:
            return {"ok": False, "error": f"Unknown category '{category}'"}
        instructions, docs = generate_recycling_instructions(category)
        return {"ok": True, "input_type": input_type, "waste_category": category,
                "recycling_instructions": instructions,
                "policy_sources": [d["source"] for d in docs]}
    except NameError as e:
        return {"ok": False, "error": f"Run Parts 2-4 first: {e}"}

In [ ]:
# text case
print(waste_management_assistant("rusty empty metal food can", input_type="text"))

In [ ]:
# image case — pick a real test image
import pathlib
c = class_names[0]; img = next((pathlib.Path(IMAGE_DIR)/c).glob("*"))
waste_management_assistant(str(img), input_type="image")

### TODO (Teddy)\n- Add more edge cases / error handling\n- Test several challenging image + text cases\n- Draw the architecture diagram; write the system evaluation